# FastVLA: Ultra-Efficient VLA Fine-Tuning on Tesla T4 (Distributed)

FastVLA is designed to democratize Vision-Language-Action (VLA) models. This notebook supports **distributed training on Kaggle's 2x T4 GPUs** with automatic mixed precision and gradient accumulation.

### Why this matters?
- Standard OpenVLA-7B (FP16) requires ~28GB VRAM (impossible on T4).
- FastVLA uses 4-bit QLoRA and Custom Triton Kernels to reduce memory by 70%.
- **Distributed Training**: Automatically utilizes both T4 GPUs on Kaggle.
- You get 6x faster iterations than standard implementations.

> **Goal:** Fine-tune OpenVLA for 50 steps on the PushT robotics dataset in under 2 minutes.

## ⚠️ Important: Gated Model Access (Llama-2)

OpenVLA internally uses **Llama-2-7B** as its language backbone. Meta requires all users to manually request access to Llama-2. 

**If you haven't been approved yet:**
1. Visit [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) and click **Request Access**.
2. **Wait for Approval**: This can take anywhere from 1 hour to 2 days.
3. **Login**: Once approved, use your HF token in the cell below.

**🚀 Want to skip the wait? (Instant Mode)**
If you want to start training **immediately** without waiting for Meta's approval, simply change the `model_id` in Step 2 to `"smolvla"`. It is non-gated and works instantly!

## 0. Authentication
1. **Add Token**: 
   - **Kaggle**: Go to **Add-ons -> Secrets**, add `HF_TOKEN` with your HuggingFace API key.
   - **Colab**: Click the **🔑 (Secrets)** icon, add `HF_TOKEN` and enable 'Notebook access'.

In [1]:
# ── 1. Setup & Installation ───────────────────────────────────────────────
# We pin numpy and upgrade huggingface_hub BEFORE importing anything
!pip install -q "numpy>=1.24.0,<2.2.0" "huggingface_hub>=0.30.0" --no-cache-dir
!pip install -q unsloth_zoo git+https://github.com/unslothai/unsloth.git --no-cache-dir
!pip install -q "fastvla[all] @ git+https://github.com/BouajilaHamza/FastVLA.git" --no-cache-dir
!pip install -q --upgrade bitsandbytes accelerate peft transformers datasets timm --no-cache-dir

print("✅ Installation complete. Please restart session if you encounter import errors.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 338.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 376.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 279.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 358.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.3/637.3 kB 382.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 326.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 283.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.8/224.8 kB 372.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 376.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 303.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 334.

## 1. Setup Environment
We install FastVLA and its optimized dependencies. 

**Tip for Kaggle users:** You can attach the 'openvla' model directly from the '+ Add Model' tab in the right sidebar to bypass the 15GB download and start training instantly!

In [2]:
# ── 2. Authentication & Verification ───────────────────────────────────────
import os
import torch
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=token)
except:
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
    except:
        login()

# Verify FastVLA
try:
    import unsloth
    print("✓ Unsloth patched")
except: pass

import fastvla
from fastvla import model as fv_model
from fastvla.kernels import TRITON_AVAILABLE

print(f"✓ FastVLA {fastvla.__version__} Ready")
print(f"  - Unsloth Optimizer: {'✓ Enabled' if fv_model.UNSLOTH_AVAILABLE else 'ℹ Disabled (using BnB fallback)'}")
print(f"  - Triton Kernels:    {'✓ Active' if TRITON_AVAILABLE else '✗ Inactive (using PyTorch fallback)'}")
print(f"  - Device:            {torch.cuda.get_device_name(0)}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✓ Unsloth patched

FastVLA Health Check
  PyTorch:    2.10.0+cu128
  CUDA:       12.8
  Device:     Tesla T4
  Unsloth:    ✓ Available
  BnB:        ✓ Available
  Triton:     ✓ Available
  GPU Memory: 0.01GB allocated, 0.02GB reserved, 0.01GB peak

✓ FastVLA 0.1.3 Ready
  - Unsloth Optimizer: ✓ Enabled
  - Triton Kernels:    ✓ Active
  - Device:            Tesla T4


## 2. Load Model in 4-bit (QLoRA)
We load OpenVLA-7B with 4-bit quantization. 

**Important:** The `action_dim` parameter must match your dataset's action dimensions. For PushT, this is 2 (x, y coordinates). For other robotics datasets, it's typically 7 (x, y, z, roll, pitch, yaw, gripper).

**Instant Start Tip:** To skip Llama-2 gated access issues, change `model_id` below to `"smolvla"`.

In [4]:
from fastvla import FastVLAModel
import torch

# Our registry recognizes "openvla-7b" and "smolvla"
# You can also pass the full HF ID "openvla/openvla-7b"
model_id = "openvla-7b" 

# IMPORTANT: action_dim must match your dataset. PushT uses 2D actions (x, y).
ACTION_DIM = 2 

print(f"Loading {model_id} in 4-bit...")
model = FastVLAModel.from_pretrained(
    model_id,
    load_in_4bit=True,
    use_peft=True,
    gradient_checkpointing=True,
    action_dim=ACTION_DIM,
)

# The new model.py includes memory diagnostics and health checks
print(f"Model loaded. {model.config.llm_name} is active.")


Loading openvla-7b in 4-bit...


TypeError: FastVLAModel.from_pretrained() takes 1 positional argument but 2 were given

## 3. Fine-Tuning on PushT (Real Robotics)
Next, we load the lerobot/pusht_image dataset and run an optimized training loop with distributed training support.

**Distributed Training Features:**
- Automatically uses both T4 GPUs on Kaggle
- Mixed precision (fp16) for faster training
- 8-bit optimizer for memory efficiency
- Gradient accumulation for larger effective batch sizes

In [5]:
from fastvla import FastVLATrainer, get_dataset

# Load only a small subset for demonstration
dataset = get_dataset("lerobot/pusht_image")

# Verify action dimensions match
sample_action = dataset[0]["actions"]
print(f"Dataset action shape: {sample_action.shape}")
print(f"Model action_dim: {model.config.action_dim}")
assert sample_action.shape[-1] == model.config.action_dim, \
    f"Action dimension mismatch! Dataset: {sample_action.shape[-1]}, Model: {model.config.action_dim}"

trainer = FastVLATrainer(
    model=model,
    dataset=dataset,
    batch_size=2,  # Per-GPU batch size
    lr=1e-4,
    max_steps=50,  # Set to 2000 for full convergence
    # Distributed training optimizations
    use_mixed_precision=True,  # fp16 for T4 GPUs
    use_8bit_optimizer=True,   # Memory efficient
    gradient_accumulation_steps=2,  # Effective batch size = 4
    # Checkpointing and logging
    output_dir="./checkpoints",
    save_steps=50,
    logging_steps=10,
)

print("Starting Distributed Training...")
results = trainer.train()

print(f"Training Complete! Final loss: {results[-1]['loss']:.4f}")

📥 Loading dataset lerobot/pusht_image from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

data/chunk-000/file-000.parquet:   0%|          | 0.00/29.8M [00:00<?, ?B/s]

data/chunk-000/file-001.parquet:   0%|          | 0.00/3.72M [00:00<?, ?B/s]

data/chunk-000/file-002.parquet:   0%|          | 0.00/3.70M [00:00<?, ?B/s]

...

data/chunk-000/file-008.parquet:   0%|          | 0.00/2.09M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/48336 [00:00<?, ? examples/s]

Dataset action shape: torch.Size([2])
Model action_dim: 2

FastVLATrainer: Mixed precision DISABLED (4-bit model detected)
  → Using direct gradient clipping (no GradScaler)
  → 4-bit models produce FP16 gradients which GradScaler cannot handle

Starting Distributed Training...


Epoch 1/1:   0%|          | 0/500 [00:00<?, ?it/s]


Unsupported: Skip calling `torch.compiler.disable()`d function
  Explanation: Skip calling function `<function AlignDevicesHook.pre_forward at 0x7cd8a447be20>` since it was wrapped with `torch.compiler.disable` (reason: None)
  Hint: Remove the `torch.compiler.disable` call

  Developer debug context: <function AlignDevicesHook.pre_forward at 0x7cd8a447be20>

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb0098.html

from user code:
   File "/kaggle/working/unsloth_compiled_cache/unsloth_compiled_module_vit.py", line 180, in ViTEmbeddings_forward
    embeddings = self.patch_embeddings(pixel_values, interpolate_pos_encoding=interpolate_pos_encoding)
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1879, in _call_impl
    return inner()

## 4. Inference (Predict Action)
Now we test the model by giving it an image and a text prompt to predict the robot's next action tensor.

In [ ]:
import numpy as np
from PIL import Image

# Dummy image for demonstration
dummy_image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
prompt = "push the t-shaped block into the goal zone"

# Process the image and prompt through the model
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

pixel_values = transform(dummy_image).unsqueeze(0).unsqueeze(0)  # [1, 1, C, H, W]
input_ids = model.tokenizer(prompt, return_tensors="pt")["input_ids"]

with torch.no_grad():
    action, _ = model(pixel_values=pixel_values, input_ids=input_ids)

print(f"Predicted Action Tensor ({model.config.action_dim}D):")
print(action)

---
**Star the repo** to support democratized robotics!
[GitHub: FastVLA](https://github.com/BouajilaHamza/FastVLA)

**Created by the FastVLA Team.**